In [ ]:
depth = 16
dim = 128 * depth
ffn_dim = 512 * depth
heads = depth
lora = 4
# vocab_size = 128000
vocab_size = 10000
length = 5
embeddings = vocab_size * dim
# embeddings = length*dim
positionnal_emb = length * dim

k = dim * dim / lora
q = dim * dim / lora
v = dim * dim / lora
o = dim * dim / lora
norm = 2 * dim
attention = heads * (k + q + v + o) + norm

ff1 = dim * dim * lora + dim * lora
ff2 = dim * dim * lora + dim
norm = 2 * dim
ffn = ff1 + ff2 + norm

transformer = depth * (attention + ffn) + embeddings + 0 * positionnal_emb
print(transformer / 10**6)

144.531456


In [2]:
from datasets import load_dataset

/home/alan/Documents/SuperQuantization/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds_train = load_dataset("codeparrot/codeparrot-train-v2-near-dedup", split="train[:10]")
ds_valid = load_dataset("codeparrot/codeparrot-valid-v2-near-dedup", split="train[:10]")

In [2]:
# import os
# os.chdir(os.path.abspath(os.path.join(os.getcwd(), '../../')))
# print(os.getcwd())
import torch
from tests.text_generation.transformer import Transformer
from super_quantization.super_quantizer import SuperQuantizer

vocab_size = 10000
seq_length = 256
depth = 4
model = Transformer(
    d_model=128*depth,
    # d_model=64,
    # d_model=4*depth,
    n_heads=depth,
    # n_heads=2,
    d_ff=512*depth,
    # d_ff=256*depth,
    # d_ff=16*depth,
    depth=depth,
    vocab_size=vocab_size,
    max_context_size=seq_length,
    lora_ratio=4
)

import collections

def analyze_model_parameters(model):
    param_groups = collections.defaultdict(float)
    total_params = 0
    
    for name, param in model.named_parameters():
        param_size = param.numel()
        total_params += param_size
        # Extraire le type de paramètre basé sur le nom
        if "self_attention.Q" in name:
            key = "Q_weights" if "weight" in name else "Q_bias"
        elif "self_attention.K" in name:
            key = "K_weights" if "weight" in name else "K_bias"
        elif "self_attention.V" in name:
            key = "V_weights" if "weight" in name else "V_bias"
        elif "self_attention.O" in name:
            key = "O_weights" if "weight" in name else "O_bias"
        elif "fc_1" in name:
            key = "fc_1_weights" if "weight" in name else "fc_1_bias"
        elif "fc_2" in name:
            key = "fc_2_weights" if "weight" in name else "fc_2_bias"
        elif "layer_norm" in name:
            key = "LayerNorm_weights" if "weight" in name else "LayerNorm_bias"
        else:
            key = name
        
        param_groups[key] += param_size
    
    # Calcul des pourcentages
    param_distribution = {k: (v / total_params) * 100 for k, v in param_groups.items()}
    
    return param_distribution

# Exemple d'utilisation avec un modèle fictif
param_percentages = analyze_model_parameters(model)
sq = SuperQuantizer()
print("Params for base:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print(sq.mesure(model)/32)

for param_type, percentage in param_percentages.items():
    print(f"{param_type}: {percentage:.2f}%")

Params for base: 17852928
17852928.0
Q_weights: 5.87%
K_weights: 5.87%
V_weights: 5.87%
O_weights: 5.87%
fc_1_weights: 23.49%
fc_1_bias: 0.05%
fc_2_weights: 23.49%
fc_2_bias: 0.01%
LayerNorm_weights: 0.02%
LayerNorm_bias: 0.02%
embedding.weight: 28.68%
position_embedding.weight: 0.73%


In [12]:
from torch import nn

fc = nn.Linear(10, 5)
fc_2 =  nn.Linear(10, 5)
fc_2.weight = fc.weight

print(id(fc.weight))
print(id(fc_2.weight))

for name, p in fc.named_parameters():
    print(name)
    print(id(p))

for name, p in fc_2.named_parameters():
    print(name)
    print(id(p))

131023229048912
131023229048912
weight
131023229048912
bias
131023229050352
weight
131023229048912
bias
131023229044688
